## NYC Property Price Prediction

Given *data about property in New York City*, let's try to predict the **price** of a given piece of property.

We will use XGBoost to make our predictions.

Data source: https://www.kaggle.com/datasets/new-york-city/nyc-property-sales

### Importing LIbraries

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import xgboost as xgb
from sklearn.metrics import r2_score

pd.set_option('display.max_columns', None)

In [2]:
data = pd.read_csv('archive/nyc-rolling-sales.csv')
data

,Unnamed: 0,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BLOCK,LOT,EASE-MENT,BUILDING CLASS AT PRESENT,ADDRESS,APARTMENT NUMBER,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE
0,4,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,392,6,,C2,153 AVENUE B,,10009,5,0,5,1633,6440,1900,2,C2,6625000,2017-07-19 00:00:00
1,5,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,26,,C7,234 EAST 4TH STREET,,10009,28,3,31,4616,18690,1900,2,C7,-,2016-12-14 00:00:00
2,6,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,39,,C7,197 EAST 3RD STREET,,10009,16,1,17,2212,7803,1900,2,C7,-,2016-12-09 00:00:00
3,7,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,402,21,,C4,154 EAST 7TH STREET,,10009,10,0,10,2272,6794,1913,2,C4,3936272,2016-09-23 00:00:00
4,8,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,404,55,,C2,301 EAST 10TH STREET,,10009,6,0,6,2369,4615,1900,2,C2,8000000,2016-11-17 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84543,8409,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7349,34,,B9,37 QUAIL LANE,,10309,2,0,2,2400,2575,1998,1,B9,450000,2016-11-28 00:00:00
84544,8410,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7349,78,,B9,32 PHEASANT LANE,,10309,2,0,2,2498,2377,1998,1,B9,550000,2017-04-21 00:00:00
84545,8411,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7351,60,,B2,49 PITNEY AVENUE,,10309,2,0,2,4000,1496,1925,1,B2,460000,2017-07-05 00:00:00
84546,8412,5,WOODROW,22 STORE BUILDINGS,4,7100,28,,K6,2730 ARTHUR KILL ROAD,,10309,0,7,7,208033,64117,2001,4,K6,11693337,2016-12-21 00:00:00


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 84548 entries, 0 to 84547
Data columns (total 22 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   Unnamed: 0                      84548 non-null  int64
 1   BOROUGH                         84548 non-null  int64
 2   NEIGHBORHOOD                    84548 non-null  str  
 3   BUILDING CLASS CATEGORY         84548 non-null  str  
 4   TAX CLASS AT PRESENT            84548 non-null  str  
 5   BLOCK                           84548 non-null  int64
 6   LOT                             84548 non-null  int64
 7   EASE-MENT                       84548 non-null  str  
 8   BUILDING CLASS AT PRESENT       84548 non-null  str  
 9   ADDRESS                         84548 non-null  str  
 10  APARTMENT NUMBER                84548 non-null  str  
 11  ZIP CODE                        84548 non-null  int64
 12  RESIDENTIAL UNITS               84548 non-null  int64
 13  COMMERCIAL U

### Preprocessing

In [24]:
data['SALE PRICE'].unique()

array([ 6625000.,  3936272.,  8000000., ...,   408092., 11693337.,
          69300.], shape=(10007,))

In [25]:
data['SALE PRICE'] = data['SALE PRICE'].replace(' -  ', np.nan).astype(float)

In [26]:
# Remove any records where we do not have a sale price
data = data.dropna(axis=0).reset_index(drop=True)

In [27]:
# Split data into X and y
y = data['SALE PRICE'].copy()
X = data.drop('SALE PRICE', axis=1).copy()

In [28]:
y

0         6625000.0
1         3936272.0
2         8000000.0
3         3192840.0
4        16232000.0
            ...    
69982      450000.0
69983      550000.0
69984      460000.0
69985    11693337.0
69986       69300.0
Name: SALE PRICE, Length: 69987, dtype: float64

In [29]:
X

,Unnamed: 0,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BLOCK,LOT,EASE-MENT,BUILDING CLASS AT PRESENT,ADDRESS,APARTMENT NUMBER,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE DATE
0,4,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,392,6,,C2,153 AVENUE B,,10009,5,0,5,1633,6440,1900,2,C2,2017-07-19 00:00:00
1,7,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,402,21,,C4,154 EAST 7TH STREET,,10009,10,0,10,2272,6794,1913,2,C4,2016-09-23 00:00:00
2,8,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,404,55,,C2,301 EAST 10TH STREET,,10009,6,0,6,2369,4615,1900,2,C2,2016-11-17 00:00:00
3,10,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,406,32,,C4,210 AVENUE B,,10009,8,0,8,1750,4226,1920,2,C4,2016-09-23 00:00:00
4,13,1,ALPHABET CITY,08 RENTALS - ELEVATOR APARTMENTS,2,387,153,,D9,629 EAST 5TH STREET,,10009,24,0,24,4489,18523,1920,2,D9,2016-11-07 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69982,8409,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7349,34,,B9,37 QUAIL LANE,,10309,2,0,2,2400,2575,1998,1,B9,2016-11-28 00:00:00
69983,8410,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7349,78,,B9,32 PHEASANT LANE,,10309,2,0,2,2498,2377,1998,1,B9,2017-04-21 00:00:00
69984,8411,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7351,60,,B2,49 PITNEY AVENUE,,10309,2,0,2,4000,1496,1925,1,B2,2017-07-05 00:00:00
69985,8412,5,WOODROW,22 STORE BUILDINGS,4,7100,28,,K6,2730 ARTHUR KILL ROAD,,10309,0,7,7,208033,64117,2001,4,K6,2016-12-21 00:00:00


In [30]:
# Remove unnecessary/difficult feature columns
{column: len(X[column].unique()) for column in X.select_dtypes('str').columns}

{'NEIGHBORHOOD': 254,
 'BUILDING CLASS CATEGORY': 47,
 'TAX CLASS AT PRESENT': 11,
 'EASE-MENT': 1,
 'BUILDING CLASS AT PRESENT': 161,
 'ADDRESS': 57304,
 'APARTMENT NUMBER': 3320,
 'LAND SQUARE FEET': 5293,
 'GROSS SQUARE FEET': 5095,
 'BUILDING CLASS AT TIME OF SALE': 161,
 'SALE DATE': 362}

In [31]:
len(X['BLOCK'].unique()), len(X['LOT'].unique())

(10848, 2484)

In [32]:
X = X.drop(['Unnamed: 0', 'BLOCK', 'EASE-MENT', 'ADDRESS', 'APARTMENT NUMBER', 'LOT'], axis=1)

In [33]:
X

,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BUILDING CLASS AT PRESENT,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE DATE
0,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,C2,10009,5,0,5,1633,6440,1900,2,C2,2017-07-19 00:00:00
1,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,C4,10009,10,0,10,2272,6794,1913,2,C4,2016-09-23 00:00:00
2,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,C2,10009,6,0,6,2369,4615,1900,2,C2,2016-11-17 00:00:00
3,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,C4,10009,8,0,8,1750,4226,1920,2,C4,2016-09-23 00:00:00
4,1,ALPHABET CITY,08 RENTALS - ELEVATOR APARTMENTS,2,D9,10009,24,0,24,4489,18523,1920,2,D9,2016-11-07 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69982,5,WOODROW,02 TWO FAMILY DWELLINGS,1,B9,10309,2,0,2,2400,2575,1998,1,B9,2016-11-28 00:00:00
69983,5,WOODROW,02 TWO FAMILY DWELLINGS,1,B9,10309,2,0,2,2498,2377,1998,1,B9,2017-04-21 00:00:00
69984,5,WOODROW,02 TWO FAMILY DWELLINGS,1,B2,10309,2,0,2,4000,1496,1925,1,B2,2017-07-05 00:00:00
69985,5,WOODROW,22 STORE BUILDINGS,4,K6,10309,0,7,7,208033,64117,2001,4,K6,2016-12-21 00:00:00


In [34]:
X['LAND SQUARE FEET'] = X['LAND SQUARE FEET'].replace(' -  ', np.nan).astype(float)

In [35]:
X['GROSS SQUARE FEET'] = X['GROSS SQUARE FEET'].replace(' -  ', np.nan).astype(float)

In [36]:
X.isna().mean()

BOROUGH                           0.000000
NEIGHBORHOOD                      0.000000
BUILDING CLASS CATEGORY           0.000000
TAX CLASS AT PRESENT              0.000000
BUILDING CLASS AT PRESENT         0.000000
ZIP CODE                          0.000000
RESIDENTIAL UNITS                 0.000000
COMMERCIAL UNITS                  0.000000
TOTAL UNITS                       0.000000
LAND SQUARE FEET                  0.302742
GROSS SQUARE FEET                 0.310615
YEAR BUILT                        0.000000
TAX CLASS AT TIME OF SALE         0.000000
BUILDING CLASS AT TIME OF SALE    0.000000
SALE DATE                         0.000000
dtype: float64

In [37]:
# Fill missing values with np.nan
for column in ['LAND SQUARE FEET', 'GROSS SQUARE FEET']:
    X[column] = X[column].fillna(X[column].mean())

In [38]:
X.isna().sum().sum()

np.int64(0)

In [39]:
X.dtypes

BOROUGH                             int64
NEIGHBORHOOD                          str
BUILDING CLASS CATEGORY               str
TAX CLASS AT PRESENT                  str
BUILDING CLASS AT PRESENT             str
ZIP CODE                            int64
RESIDENTIAL UNITS                   int64
COMMERCIAL UNITS                    int64
TOTAL UNITS                         int64
LAND SQUARE FEET                  float64
GROSS SQUARE FEET                 float64
YEAR BUILT                          int64
TAX CLASS AT TIME OF SALE           int64
BUILDING CLASS AT TIME OF SALE        str
SALE DATE                             str
dtype: object

In [40]:
# Get year, month and day features from SALE DATE column
X['SALE DATE'] = pd.to_datetime(X['SALE DATE'])

X['YEAR'] = X['SALE DATE'].apply(lambda x: x.year)
X['MONTH'] = X['SALE DATE'].apply(lambda x: x.month)
X['DAY'] = X['SALE DATE'].apply(lambda x: x.day)

X = X.drop('SALE DATE', axis=1)

In [41]:
X

,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BUILDING CLASS AT PRESENT,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,YEAR,MONTH,DAY
0,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,C2,10009,5,0,5,1633.0,6440.0,1900,2,C2,2017,7,19
1,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,C4,10009,10,0,10,2272.0,6794.0,1913,2,C4,2016,9,23
2,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,C2,10009,6,0,6,2369.0,4615.0,1900,2,C2,2016,11,17
3,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,C4,10009,8,0,8,1750.0,4226.0,1920,2,C4,2016,9,23
4,1,ALPHABET CITY,08 RENTALS - ELEVATOR APARTMENTS,2,D9,10009,24,0,24,4489.0,18523.0,1920,2,D9,2016,11,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69982,5,WOODROW,02 TWO FAMILY DWELLINGS,1,B9,10309,2,0,2,2400.0,2575.0,1998,1,B9,2016,11,28
69983,5,WOODROW,02 TWO FAMILY DWELLINGS,1,B9,10309,2,0,2,2498.0,2377.0,1998,1,B9,2017,4,21
69984,5,WOODROW,02 TWO FAMILY DWELLINGS,1,B2,10309,2,0,2,4000.0,1496.0,1925,1,B2,2017,7,5
69985,5,WOODROW,22 STORE BUILDINGS,4,K6,10309,0,7,7,208033.0,64117.0,2001,4,K6,2016,12,21


In [43]:
# Make numeric categorical features into string columns
for column in ['BOROUGH', 'ZIP CODE']:
    X[column] = X[column].astype(str)

In [44]:
{column: len(X[column].unique()) for column in X.select_dtypes('str').columns}

{'BOROUGH': 5,
 'NEIGHBORHOOD': 254,
 'BUILDING CLASS CATEGORY': 47,
 'TAX CLASS AT PRESENT': 11,
 'BUILDING CLASS AT PRESENT': 161,
 'ZIP CODE': 184,
 'BUILDING CLASS AT TIME OF SALE': 161}

In [45]:
def onehot_encode(df, columns, prefixes):
    df = df.copy()
    for column, prefix in zip(columns, prefixes):
        dummies = pd.get_dummies(df[column], prefix=prefix, dtype=int)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(column, axis=1)
    return df

In [46]:
# One hot encode remaining nominal columns
nominal_features = ['BOROUGH', 'NEIGHBORHOOD', 'BUILDING CLASS CATEGORY', 'TAX CLASS AT PRESENT', 
                   'BUILDING CLASS AT PRESENT', 'ZIP CODE', 'BUILDING CLASS AT TIME OF SALE']

nominal_prefixes = ['B', 'N', 'BCC', 'TCP', 'BCP', 'ZC', 'BCTS']

X = onehot_encode(X, nominal_features, nominal_prefixes)

In [47]:
X

,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,YEAR,MONTH,DAY,B_1,B_2,B_3,B_4,B_5,N_AIRPORT LA GUARDIA,N_ALPHABET CITY,N_ANNADALE,N_ARDEN HEIGHTS,N_ARROCHAR,N_ARROCHAR-SHORE ACRES,N_ARVERNE,N_ASTORIA,N_BATH BEACH,N_BATHGATE,N_BAY RIDGE,N_BAYCHESTER,N_BAYSIDE,N_BEDFORD PARK/NORWOOD,N_BEDFORD STUYVESANT,N_BEECHHURST,N_BELLE HARBOR,N_BELLEROSE,N_BELMONT,N_BENSONHURST,N_BERGEN BEACH,N_BLOOMFIELD,N_BOERUM HILL,N_BOROUGH PARK,N_BRIARWOOD,N_BRIGHTON BEACH,N_BROAD CHANNEL,N_BRONX PARK,N_BRONXDALE,N_BROOKLYN HEIGHTS,N_BROWNSVILLE,N_BULLS HEAD,N_BUSH TERMINAL,N_BUSHWICK,N_CAMBRIA HEIGHTS,N_CANARSIE,N_CARROLL GARDENS,N_CASTLE HILL/UNIONPORT,N_CASTLETON CORNERS,N_CHELSEA,N_CHINATOWN,N_CITY ISLAND,N_CITY ISLAND-PELHAM STRIP,N_CIVIC CENTER,N_CLINTON,N_CLINTON HILL,N_CLOVE LAKES,N_CO-OP CITY,N_COBBLE HILL,N_COBBLE HILL-WEST,N_COLLEGE POINT,N_CONCORD,N_CONCORD-FOX HILLS,N_CONEY ISLAND,N_CORONA,N_COUNTRY CLUB,N_CROTONA PARK,N_CROWN HEIGHTS,N_CYPRESS HILLS,N_DONGAN HILLS,N_DONGAN HILLS-COLONY,N_DONGAN HILLS-OLD TOWN,N_DOUGLASTON,N_DOWNTOWN-FULTON FERRY,N_DOWNTOWN-FULTON MALL,N_DOWNTOWN-METROTECH,N_DYKER HEIGHTS,N_EAST ELMHURST,N_EAST NEW YORK,N_EAST RIVER,N_EAST TREMONT,N_EAST VILLAGE,N_ELMHURST,N_ELTINGVILLE,N_EMERSON HILL,N_FAR ROCKAWAY,N_FASHION,N_FIELDSTON,N_FINANCIAL,N_FLATBUSH-CENTRAL,N_FLATBUSH-EAST,N_FLATBUSH-LEFFERTS GARDEN,N_FLATBUSH-NORTH,N_FLATIRON,N_FLATLANDS,N_FLORAL PARK,N_FLUSHING MEADOW PARK,N_FLUSHING-NORTH,N_FLUSHING-SOUTH,N_FORDHAM,N_FOREST HILLS,N_FORT GREENE,N_FRESH KILLS,N_FRESH MEADOWS,N_GERRITSEN BEACH,N_GLEN OAKS,N_GLENDALE,N_GOWANUS,N_GRAMERCY,N_GRANT CITY,N_GRASMERE,N_GRAVESEND,N_GREAT KILLS,N_GREAT KILLS-BAY TERRACE,N_GREENPOINT,N_GREENWICH VILLAGE-CENTRAL,N_GREENWICH VILLAGE-WEST,N_GRYMES HILL,N_HAMMELS,N_HARLEM-CENTRAL,N_HARLEM-EAST,N_HARLEM-UPPER,N_HARLEM-WEST,N_HIGHBRIDGE/MORRIS HEIGHTS,N_HILLCREST,N_HOLLIS,N_HOLLIS HILLS,N_HOLLISWOOD,N_HOWARD BEACH,N_HUGUENOT,N_HUNTS POINT,N_INWOOD,N_JACKSON HEIGHTS,N_JAMAICA,N_JAMAICA BAY,N_JAMAICA ESTATES,N_JAMAICA HILLS,N_JAVITS CENTER,N_KENSINGTON,N_KEW GARDENS,N_KINGSBRIDGE HTS/UNIV HTS,N_KINGSBRIDGE/JEROME PARK,N_KIPS BAY,N_LAURELTON,N_LITTLE ITALY,N_LITTLE NECK,N_LIVINGSTON,N_LONG ISLAND CITY,N_LOWER EAST SIDE,N_MADISON,N_MANHATTAN BEACH,N_MANHATTAN VALLEY,N_MANOR HEIGHTS,N_MARINE PARK,N_MARINERS HARBOR,N_MASPETH,N_MELROSE/CONCOURSE,N_MIDDLE VILLAGE,N_MIDLAND BEACH,N_MIDTOWN CBD,N_MIDTOWN EAST,N_MIDTOWN WEST,N_MIDWOOD,N_MILL BASIN,N_MORNINGSIDE HEIGHTS,N_MORRIS PARK/VAN NEST,N_MORRISANIA/LONGWOOD,N_MOTT HAVEN/PORT MORRIS,N_MOUNT HOPE/MOUNT EDEN,N_MURRAY HILL,N_NAVY YARD,N_NEPONSIT,N_NEW BRIGHTON,N_NEW BRIGHTON-ST. GEORGE,N_NEW DORP,N_NEW DORP-BEACH,N_NEW DORP-HEIGHTS,N_NEW SPRINGVILLE,N_OAKLAND GARDENS,N_OAKWOOD,N_OAKWOOD-BEACH,N_OCEAN HILL,N_OCEAN PARKWAY-NORTH,N_OCEAN PARKWAY-SOUTH,N_OLD MILL BASIN,N_OZONE PARK,N_PARK SLOPE,N_PARK SLOPE SOUTH,N_PARKCHESTER,N_PELHAM BAY,N_PELHAM GARDENS,N_PELHAM PARKWAY NORTH,N_PELHAM PARKWAY SOUTH,N_PLEASANT PLAINS,N_PORT IVORY,N_PORT RICHMOND,N_PRINCES BAY,N_PROSPECT HEIGHTS,N_QUEENS VILLAGE,N_RED HOOK,N_REGO PARK,N_RICHMOND HILL,N_RICHMONDTOWN,N_RICHMONDTOWN-LIGHTHS HILL,N_RIDGEWOOD,N_RIVERDALE,N_ROCKAWAY PARK,N_ROOSEVELT ISLAND,N_ROSEBANK,N_ROSEDALE,N_ROSSVILLE,N_ROSSVILLE-CHARLESTON,N_ROSSVILLE-PORT MOBIL,N_ROSSVILLE-RICHMOND VALLEY,N_SCHUYLERVILLE/PELHAM BAY,N_SEAGATE,N_SHEEPSHEAD BAY,N_SILVER LAKE,N_SO. JAMAICA-BAISLEY PARK,N_SOHO,N_SOUNDVIEW,N_SOUTH BEACH,N_SOUTH JAMAICA,N_SOUTH OZONE PARK,N_SOUTHBRIDGE,N_SPRING CREEK,N_SPRINGFIELD GARDENS,N_ST. ALBANS,N_STAPLETON,N_STAPLETON-CLIFTON,N_SUNNYSIDE,N_SUNSET PARK,N_THROGS NECK,N_TODT HILL,N_TOMPKINSVILLE,N_TOTTENVILLE,N_TRAVIS,N_TRIBECA,N_UPPER EAST SIDE (59-79),N_UPPER EAST SIDE (79-96),N_UPPER EAST SIDE (96-110),N_UPPER WEST SIDE (59-79),N_UPPER WEST SIDE (79-96),N_UPPER WEST SIDE (96-116),N_VAN CORTLANDT PARK,N_WAKEFIELD,N_WASHINGTON HEIGHTS LOWER,N_WASHINGTON HEIGHTS UPPER,N_WEST NEW BRIGHTON,N_WESTCHESTER,

In [48]:
y

0         6625000.0
1         3936272.0
2         8000000.0
3         3192840.0
4        16232000.0
            ...    
69982      450000.0
69983      550000.0
69984      460000.0
69985    11693337.0
69986       69300.0
Name: SALE PRICE, Length: 69987, dtype: float64

In [49]:
# Scale X with a standard scaler
scaler = StandardScaler()

X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

### Training

In [52]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=123)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, train_size=0.7, random_state=123)

In [53]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

In [55]:
params = {'learning_rate': 0.001, 'max_depth': 6, 'lambda': 0.01}

model = xgb.train(params, dtrain, num_boost_round=10000, evals=[(dval, 'eval')], early_stopping_rounds=10)

[0]	eval-rmse:6350362.70889
[1]	eval-rmse:6349248.69355
[2]	eval-rmse:6348144.20980
[3]	eval-rmse:6347049.31016
[4]	eval-rmse:6345963.93568
[5]	eval-rmse:6344887.94717
[6]	eval-rmse:6343821.43550
[7]	eval-rmse:6342764.32470
[8]	eval-rmse:6341716.58966
[9]	eval-rmse:6340678.35005
[10]	eval-rmse:6339649.34158
[11]	eval-rmse:6338629.63553
[12]	eval-rmse:6337619.31223
[13]	eval-rmse:6336618.14126
[14]	eval-rmse:6335626.33901
[15]	eval-rmse:6334643.89021
[16]	eval-rmse:6333670.51865
[17]	eval-rmse:6332706.22044
[18]	eval-rmse:6331751.28429
[19]	eval-rmse:6330805.49344
[20]	eval-rmse:6329868.74817
[21]	eval-rmse:6328941.25676
[22]	eval-rmse:6328022.76357
[23]	eval-rmse:6327113.31700
[24]	eval-rmse:6326213.02110
[25]	eval-rmse:6325321.68369
[26]	eval-rmse:6324439.31732
[27]	eval-rmse:6323565.86446
[28]	eval-rmse:6322701.59194
[29]	eval-rmse:6321846.10860
[30]	eval-rmse:6320999.62601
[31]	eval-rmse:6320161.94304
[32]	eval-rmse:6319333.30268
[33]	eval-rmse:6318513.28773
[34]	eval-rmse:6317702.2

In [56]:
y_true = np.array(y_test)
y_pred = model.predict(dtest)

In [57]:
print("Model R^2 Score: {:.4f}".format(r2_score(y_true, y_pred)))

Model R^2 Score: 0.0554
